# Time/Attitude/Depth-Segmented Spectral Fit

This notebook demonstrates how to add a **temporal dimension** to the spectral-fit likelihood in cosipy.

The standard spectral-fit tutorials (see `continuum_fit` and `extended_source_fit`) bin the data in three dimensions: measured energy (`Em`), Compton scattering angle (`Phi`), and scattering direction (`PsiChi`) -- the Compton Data Space (CDS). The response and background are time-integrated over the whole observation, and the fit has no notion of *when* (or from which spacecraft attitude) each count was recorded.

This notebook shows how to add one extra axis to that likelihood, so the fit can account for how the source response, background, or simply the data itself vary with:

- **Time** (e.g. a light curve binning), or
- **spacecraft attitude** (`ScAtt`, same binning concept used by the local-coordinate image-deconvolution tutorials, see `docs/tutorials/image_deconvolution/optional/scatt_binned_data_w_local_coordinate`), or
- any other **user-defined quantity** recorded per event, when only the background (not the response) depends on it -- e.g. **atmospheric depth/overburden** for a balloon payload (where the atmospheric background rate and shape change substantially during ascent, float, and descent), or any other per-event quantity you care to track for a satellite instrument.

We use the small response/orientation files that ship with cosipy's test suite, so this notebook runs instantly with **no large downloads**. For a real analysis, swap in your own `FullDetectorResponse` and orientation (`SpacecraftHistory`/`SpacecraftFile`) files -- the API is identical.

**Contents**
1. Time-segmented **point-source** spectral fit
2. Time-segmented **extended/diffuse-source** spectral fit
3. Swapping `Time` for spacecraft-attitude (`ScAtt`) segmentation
4. Segmenting on a response-independent axis (e.g. `Depth`), where only the background varies


In [ ]:
import logging
import sys

logger = logging.getLogger('cosipy')
logger.setLevel(logging.INFO)
logger.addHandler(logging.StreamHandler(sys.stdout))

import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord

import matplotlib.pyplot as plt
%matplotlib inline

from histpy import Histogram, Axis, Axes

from threeML import Powerlaw, PointSource, ExtendedSource, Model, JointLikelihood, DataList
from astromodels import Parameter, Gaussian_on_sphere

from cosipy import test_data
from cosipy.data_io import SegmentedEmCDSBinnedData
from cosipy.spacecraftfile import SpacecraftHistory
from cosipy.image_deconvolution import TimeBinnedExposureTable, SpacecraftAttitudeExposureTable, CoordsysConversionMatrix
from cosipy.response import (
    FullDetectorResponse,
    BinnedInstrumentResponse,
    PointSourceResponse,
    BinnedThreeMLModelFolding,
    BinnedThreeMLPointSourceResponse,
    BinnedThreeMLSegmentedPointSourceResponse,
    BinnedThreeMLSegmentedExtendedSourceResponse,
)
from cosipy.background_estimation import FreeNormBinnedBackground
from cosipy.interfaces import ThreeMLPluginInterface
from cosipy.statistics import PoissonLikelihood


## 1. Time-segmented point-source spectral fit

### Load a detector response and an orientation (attitude) history

We use the same small test files used throughout cosipy's own test suite. They cover only 10 seconds of (simulated) orientation, which is enough to demonstrate the API.

In [ ]:
dr_path = str(test_data.path / "test_full_detector_response.h5")
fdr = FullDetectorResponse.open(dr_path)

ori = SpacecraftHistory.open(test_data.path / "20280301_first_10sec.fits")

print(f"Response axes: {fdr.axes.labels}")
print(f"Orientation spans {(ori.obstime[-1] - ori.obstime[0]).to(u.s)}, with {len(ori.obstime)} samples.")


### Build a `Time`-binned exposure table

`TimeBinnedExposureTable` (and its spacecraft-attitude counterpart, `SpacecraftAttitudeExposureTable`) come from the image-deconvolution module, where they're used to bin ScAtt/Time-resolved data for local-coordinate imaging (see the `scatt_binned_data_w_local_coordinate` and `time_binned_data_w_local_coordinate` tutorials). We reuse the *same* exposure-table infrastructure here for spectral fitting: each row carries a time-averaged attitude and a livetime, and the **same** exposure table is used both to bin the data and to build the matching segmented response, so the two stay consistent.

Here we split the 10 s of orientation data into 4 equal `Time` bins.

In [ ]:
n_segments = 4

t0 = ori.obstime[0]
t1 = ori.obstime[-1]
edges = Time(np.linspace(t0.unix, t1.unix, n_segments + 1), format='unix', scale=t0.scale)

exposure_table = TimeBinnedExposureTable.from_orientation(ori,
                                                           tstart_list=edges[:-1],
                                                           tstop_list=edges[1:])

exposure_table[['tstart', 'tstop', 'total_livetime']]


### Build the segmented data skeleton and instrument response

The data histogram now has **four** axes: the segment axis (`Time` here) plus the usual `Em`, `Phi`, `PsiChi`. This is exactly what `SegmentedEmCDSBinnedData` validates.

We fit here in Galactic (inertial) coordinates, matching the usual point-source spectral-fit workflow (see `continuum_fit`): the `PsiChi` axis is in the galactic frame, and the response is recomputed, per `Time` bin, from the spacecraft attitude recorded in `exposure_table`.

In [ ]:
l, b = 50, -45
source_coord = SkyCoord(l=l * u.deg, b=b * u.deg, frame='galactic')

psichi_axis_gal = fdr.axes['PsiChi'].copy()
psichi_axis_gal.coordsys = 'galactic'

time_axis = Axis(np.arange(n_segments + 1), label='Time')
data_axes = Axes([time_axis, fdr.axes['Em'], fdr.axes['Phi'], psichi_axis_gal], copy_axes=False)

# Empty skeleton: only used to fix the axes/binning, not real counts yet.
skeleton = SegmentedEmCDSBinnedData(Histogram(data_axes))

instrument_response = BinnedInstrumentResponse(fdr, skeleton.to_emcds())


### Inject a known source + background to fit

In a real analysis, `data` below would come from your own measured events, binned with `exposure_table.get_binned_data(unbinned_data, psichi_binning='galactic')` (or `BinnedData.get_segmented_em_cds('Time')` if you already bin a `Time` axis the usual way). Here, to demonstrate (and validate) that the fit recovers known inputs, we inject a fake power-law point source plus a flat background using `PointSourceResponse.from_exposure_table` -- the same segmented-response machinery the fit itself will use.

In [ ]:
K_true = 1 / u.cm / u.cm / u.s / u.keV
index_true = -2
piv = 1 * u.MeV

spectrum = Powerlaw()
spectrum.K.value = K_true.value
spectrum.K.unit = K_true.unit
spectrum.index.value = index_true
spectrum.piv.value = piv.value
spectrum.piv.unit = piv.unit

source = PointSource("source", l=l, b=b, spectral_shape=spectrum)

psr_truth = PointSourceResponse.from_exposure_table(source_coord,
                                                    skeleton,
                                                    instrument_response,
                                                    exposure_table,
                                                    energy_axis=fdr.axes['Ei'])

data_dist = psr_truth.get_expectation(source.spectrum.main.shape)

bkg_dist = data_dist.copy()
bkg_dist[:] = np.mean(bkg_dist.contents)  # flat background, same level in every Time bin
data_dist += bkg_dist

bkg_rate = np.sum(bkg_dist.contents) / ori.cumulative_livetime()
print(f"Injected background rate: {bkg_rate.to(u.Hz)}")


### A quick look at the segmented data

Projecting the 4D histogram onto the `Time` axis gives a light curve -- this is the new information this axis adds to the fit.

In [ ]:
lightcurve = data_dist.project('Time').to_dense(copy=False)

plt.step(np.arange(n_segments), lightcurve.contents, where='mid')
plt.xlabel("Time bin")
plt.ylabel("Counts")
plt.title("Injected (source + background) counts per Time bin")
plt.show()


### Set up and run the segmented fit

This mirrors the usual COSI 3ML plugin setup (see `continuum_fit`), with two differences:

- `data`/`bkg` are built from `SegmentedEmCDSBinnedData`/a background histogram with the extra `Time` axis.
- The point-source response is `BinnedThreeMLSegmentedPointSourceResponse` instead of `BinnedThreeMLPointSourceResponse`, and takes `exposure_table` instead of `sc_history` directly -- it recomputes the response per `Time` bin using that table's per-bin attitude.

Everything downstream (`BinnedThreeMLModelFolding`, `PoissonLikelihood`, `FreeNormBinnedBackground`, `ThreeMLPluginInterface`) is completely unchanged: they work transparently with the extra axis.

In [ ]:
# Move the initial guess away from the injected values, to make sure the fit has to do real work
spectrum.index.value = index_true * 1.1
spectrum.K.value = K_true.value * 1.1

data = SegmentedEmCDSBinnedData(data_dist)
bkg = FreeNormBinnedBackground(bkg_dist, sc_history=ori, copy=False)

psr = BinnedThreeMLSegmentedPointSourceResponse(data=data,
                                                instrument_response=instrument_response,
                                                exposure_table=exposure_table,
                                                energy_axis=fdr.axes['Ei'],
                                                polarization_axis=fdr.axes['Pol'] if 'Pol' in fdr.axes.labels else None)

response = BinnedThreeMLModelFolding(data=data, point_source_response=psr)

like_fun = PoissonLikelihood(data, response, bkg)

cosi = ThreeMLPluginInterface('cosi', like_fun, response, bkg)

cosi.bkg_parameter['bkg_norm'] = Parameter("bkg_norm",
                                           1.1 * bkg_rate.to_value(u.Hz),
                                           unit=u.Hz,
                                           min_value=0,
                                           max_value=1e6,
                                           delta=1e5)

plugins = DataList(cosi)
model = Model(source)

like = JointLikelihood(model, plugins, verbose=False)
like.fit(compute_covariance=False)


In [ ]:
print("Fitted K:        ", spectrum.K.value, " (true:", K_true.value, ")")
print("Fitted index:    ", spectrum.index.value, " (true:", index_true, ")")
print("Fitted bkg_norm: ", cosi.bkg_parameter['bkg_norm'].value, " Hz (true:", bkg_rate.to_value(u.Hz), ")")


## 2. Time-segmented extended/diffuse-source spectral fit

For an extended (diffuse) source, the non-segmented workflow (see `extended_source_fit`) folds the sky model through a **precomputed**, Galactic-frame `ExtendedSourceResponse` file. That file already has the spacecraft attitude integrated over the *entire* observation baked in, so it cannot be split into Time/ScAtt segments without being recomputed.

Instead, the segmented extended-source response is built directly from the **local-frame** `FullDetectorResponse` and a `CoordsysConversionMatrix` (CCM) -- exactly the forward model used by the local-coordinate (ScAtt/Time) image-deconvolution tutorials, specialized here to a *fixed* astromodels sky model instead of a free-form image. The CCM carries, per segment bin, the dwell time from each sky pixel into each local detector pixel.

Because this goes through the local frame, `PsiChi` must be in the spacecraft frame here (unlike the point-source case above, which used the galactic frame).

In [ ]:
ccm = CoordsysConversionMatrix.from_exposure_table(exposure_table, fdr,
                                                   nside_model=fdr.nside,
                                                   use_averaged_pointing=False)

ccm


In [ ]:
spectrum_ext = Powerlaw()
spectrum_ext.K.value = 1e-3
spectrum_ext.K.unit = 1 / u.cm / u.cm / u.s / u.keV
spectrum_ext.piv.value = 1000
spectrum_ext.piv.unit = u.keV
spectrum_ext.index.value = -2

morphology = Gaussian_on_sphere(lon0=l, lat0=b, sigma=10)
morphology.lon0.free = False
morphology.lat0.free = False
morphology.sigma.free = False

ext_source = ExtendedSource('diffuse', spatial_shape=morphology, spectral_shape=spectrum_ext)

time_axis_local = Axis(np.arange(n_segments + 1), label='Time')
local_data_axes = Axes([time_axis_local, fdr.axes['Em'], fdr.axes['Phi'], fdr.axes['PsiChi']], copy_axes=False)
local_skeleton = SegmentedEmCDSBinnedData(Histogram(local_data_axes))

esr_truth = BinnedThreeMLSegmentedExtendedSourceResponse(local_skeleton, fdr, ccm)
esr_truth.set_source(ext_source)
ext_data_dist = esr_truth.expectation()

ext_bkg_dist = ext_data_dist.copy()
ext_bkg_dist[:] = np.mean(ext_data_dist.contents)
ext_data_dist += ext_bkg_dist

ext_bkg_rate = np.sum(ext_bkg_dist.contents) / ori.cumulative_livetime()
print(f"Injected background rate: {ext_bkg_rate.to(u.Hz)}")


In [ ]:
# Move the initial guess away from the injected values
spectrum_ext.index.value = -2 * 1.1
spectrum_ext.K.value = 1e-3 * 1.1

ext_data = SegmentedEmCDSBinnedData(ext_data_dist)
ext_bkg = FreeNormBinnedBackground(ext_bkg_dist, sc_history=ori, copy=False)

esr = BinnedThreeMLSegmentedExtendedSourceResponse(ext_data, fdr, ccm)

ext_response = BinnedThreeMLModelFolding(data=ext_data, extended_source_response=esr)

ext_like_fun = PoissonLikelihood(ext_data, ext_response, ext_bkg)

ext_cosi = ThreeMLPluginInterface('cosi_ext', ext_like_fun, ext_response, ext_bkg)

ext_cosi.bkg_parameter['bkg_norm'] = Parameter("bkg_norm",
                                               1.1 * ext_bkg_rate.to_value(u.Hz),
                                               unit=u.Hz,
                                               min_value=0,
                                               max_value=1e6,
                                               delta=1e5)

ext_plugins = DataList(ext_cosi)
ext_model = Model(ext_source)

ext_like = JointLikelihood(ext_model, ext_plugins, verbose=False)
ext_like.fit(compute_covariance=False)


In [ ]:
print("Fitted K:        ", spectrum_ext.K.value, " (true:", 1e-3, ")")
print("Fitted index:    ", spectrum_ext.index.value, " (true:", -2, ")")
print("Fitted bkg_norm: ", ext_cosi.bkg_parameter['bkg_norm'].value, " Hz (true:", ext_bkg_rate.to_value(u.Hz), ")")


## 3. Using spacecraft-attitude (`ScAtt`) segmentation instead of `Time`

Everything above works unchanged if you replace `Time` binning with spacecraft-attitude (`ScAtt`) binning: just build a `SpacecraftAttitudeExposureTable` instead of a `TimeBinnedExposureTable`. Both are subclasses of the same `ExposureTableBase`, and expose the same per-row interface (`zpointing_averaged`, `xpointing_averaged`, `total_livetime`), so none of the fitting code above needs to change -- only how `exposure_table` is built.

This is the natural choice if you have many short/scattered exposures and care about the attitude-dependence of the response more than the absolute time (e.g. a source observed over a full precessing orbit).

In [ ]:
# Use a finer nside than the response's own (just for the attitude
# binning -- it doesn't need to match the response/image nside) so that
# the spacecraft's small motion over these 10 s resolves into more than
# one attitude bin.
scatt_exposure_table = SpacecraftAttitudeExposureTable.from_orientation(ori,
                                                                        nside=64,
                                                                        start=ori.obstime[0],
                                                                        stop=ori.obstime[-1],
                                                                        min_livetime=0,
                                                                        min_num_pointings=1)

print(f"{len(scatt_exposure_table)} unique spacecraft-attitude bins over the 10 s file.")

# From here on, everything is identical to Part 1/2: e.g. for the point-source case,
#   psr = BinnedThreeMLSegmentedPointSourceResponse(data=..., instrument_response=...,
#                                                    exposure_table=scatt_exposure_table, ...)
# just using `scatt_exposure_table` (and a data/CCM 'ScAtt' axis) instead of the 'Time' one.


## 4. Segmenting on a response-independent axis (e.g. `Depth`)

Sometimes the extra axis isn't time or attitude at all, but some other per-event quantity whose main effect is on the **background** rather than on the instrument response. A concrete motivating case is a **balloon-borne payload**: the atmospheric depth/overburden above the instrument changes continuously during ascent, float, and descent, and the atmospheric gamma-ray background -- both its normalization and its spectral shape -- varies substantially with it, while the instrument's response to a source photon of a given energy and direction does not. (For a satellite, the same broadcasting mechanism applies to whatever other per-event quantity you want to track, if the response itself doesn't depend on it.)

In that case there's no need for a segmented response at all: build `data` and the background template with the extra axis (here labeled `Depth`, standing in for atmospheric depth -- label it however makes sense for your analysis, e.g. `AtmDepth`), but keep using the ordinary (non-segmented) point-/extended-source response. `BinnedThreeMLModelFolding` automatically **broadcasts** a response that doesn't carry the extra axis across all of its bins.

The example below builds a synthetic `Depth` axis with 3 bins (e.g. standing in for three atmospheric-depth bins spanning ascent to float) where only the background level differs; the point-source response itself stays the ordinary, non-segmented `BinnedThreeMLPointSourceResponse`, since the detector response to a given photon doesn't change with atmospheric depth.

In [ ]:
n_depth = 3

# Ordinary (non-segmented) point-source response, exactly as in the standard
# continuum_fit tutorial.
instrument_response_plain = BinnedInstrumentResponse(fdr, skeleton.to_emcds())

psr_plain = BinnedThreeMLPointSourceResponse(data=skeleton.to_emcds(),
                                             instrument_response=instrument_response_plain,
                                             sc_history=ori,
                                             energy_axis=fdr.axes['Ei'],
                                             polarization_axis=fdr.axes['Pol'] if 'Pol' in fdr.axes.labels else None,
                                             nside=2 * fdr.nside)

psr_plain.set_source(source)
source_expectation = psr_plain.expectation()  # axes: (Em, Phi, PsiChi) -- no Depth axis at all

# Build a Depth-segmented data set (e.g. 3 atmospheric-depth bins for a balloon
# payload) where the background differs bin to bin, but the source contribution
# is identical in every Depth bin.
depth_axis = Axis(np.arange(n_depth + 1), label='Depth')
depth_axes = Axes([depth_axis, fdr.axes['Em'], fdr.axes['Phi'], psichi_axis_gal], copy_axes=False)

depth_bkg_dist = Histogram(depth_axes)
for i, level in enumerate([0.5, 1.0, 2.0]):  # e.g. background rising with atmospheric overburden
    depth_bkg_dist[i] = level * np.mean(source_expectation.contents)

depth_data_dist = Histogram(depth_axes)
for i in range(n_depth):
    depth_data_dist[i] = source_expectation.contents + depth_bkg_dist[i]

depth_bkg_rate = np.sum(depth_bkg_dist.contents) / ori.cumulative_livetime()

depth_data = SegmentedEmCDSBinnedData(depth_data_dist)
depth_bkg = FreeNormBinnedBackground(depth_bkg_dist, sc_history=ori, copy=False)

# Note: point_source_response is the *plain*, non-segmented psr_plain --
# BinnedThreeMLModelFolding broadcasts it across all 3 Depth bins automatically.
depth_response = BinnedThreeMLModelFolding(data=depth_data, point_source_response=psr_plain)
depth_response.set_model(Model(source))

depth_expectation = depth_response.expectation()
print(f"Expectation axes: {depth_expectation.axes.labels}")

# Every Depth slice of the *source* contribution is identical, since the response
# doesn't depend on Depth -- only the injected background (and hence total counts) differs.
depth_light_curve = depth_data_dist.project('Depth').to_dense(copy=False)
plt.step(np.arange(n_depth), depth_light_curve.contents, where='mid')
plt.xlabel("Depth bin")
plt.ylabel("Counts")
plt.title("Counts per Depth bin (e.g. atmospheric overburden; background-only variation)")
plt.show()


## Summary

New building blocks introduced for segmented spectral fitting:

- `cosipy.data_io.SegmentedEmCDSBinnedData`: data (and background) with an extra `Em`/`Phi`/`PsiChi` + segment axis (`Time`, `ScAtt`, or any custom label).
- `cosipy.response.PointSourceResponse.from_exposure_table`: a segmented point-source response, one slice per exposure-table row.
- `cosipy.response.BinnedThreeMLSegmentedPointSourceResponse`: the live/fit-time version used as `point_source_response=` in `BinnedThreeMLModelFolding`.
- `cosipy.response.BinnedThreeMLSegmentedExtendedSourceResponse`: the extended/diffuse-source analog, built from a `FullDetectorResponse` + `CoordsysConversionMatrix` instead of a precomputed Galactic-frame response file.
- `BinnedThreeMLModelFolding` automatically **broadcasts** a non-segmented response across an extra axis that only the background/data use (the `Depth`/atmospheric-overburden example above).

The exposure-table binning (`TimeBinnedExposureTable`/`SpacecraftAttitudeExposureTable`) is shared with the local-coordinate image-deconvolution tutorials -- see `docs/tutorials/image_deconvolution/optional/time_binned_data_w_local_coordinate` and `.../scatt_binned_data_w_local_coordinate` for how to build these exposure tables (and bin real data against them) for a full-scale observation.

See also the test suite for more worked examples:
`tests/threeml/test_segmented_spectral_fitting.py`, `tests/threeml/test_segmented_extended_source_spectral_fitting.py`, `tests/response/test_point_source_response.py`, `tests/response/test_segmented_extended_source_response.py`, and `tests/response/test_model_folding_broadcast.py`.
